# CPU Overheads

In [13]:
import os
import re
import json
from common import *
from collections import defaultdict
from data import RawDataFile
from experiment import Treatment, NetworkSetting
from treatments.picoquic import treatment_map as http_treatment_map
from treatments.media import treatment_map as media_treatment_map

## Collect data

```
cd deps
./build_deps.sh [7|9]
cd ..
sudo -E python3 emulation/main.py --bw1 20 --bw2 20 --delay2 100 --loss1 4 --loss2 0 \
--proxy sidekick --freq-pkts 16 --freq-ms 10 --threshold 160 --riblt --quack-hint \
picoquic --client-quacker --ack-delay 30 -n 25000000 
```

In [20]:
# Make the data directory
home_base = f'{DATA_HOME}/cycles_base/'
home_quack = f'{DATA_HOME}/cycles_quack/'
os.system(f'mkdir -p {home_base}')
os.system(f'mkdir -p {home_quack}')

0

In [23]:
# HTTP benchmark
def gen_http_cmd(label, time_s, base: bool):
    if base:
        print('./deps/build_deps.sh 7')
        home = home_base
    else:
        print('./deps/build_deps.sh 9')
        home = home_quack
    treatment = http_treatment_map[label]
    network_setting = NetworkSetting(bw1=20, bw2=20, delay1=1, delay2=100, loss1=4, loss2=0)
    data_size = 125000 * 20 * time_s # 125000 * bottleneck_bw * time_s
    cmd = RawDataFile(treatment, network_setting, '').cmd(data_size=data_size, num_trials=1, logdir=home)
    print(cmd)

gen_http_cmd('picoquic_iblt_30ms_hint', 10, base=True)
gen_http_cmd('picoquic_iblt_30ms_hint', 10, base=False)

./deps/build_deps.sh 7
sudo -E python3 emulation/main.py --bw1 20 --bw2 20 --delay2 100 --loss1 4 --loss2 0 -t 1 --logdir /home/ubuntu/sidekick-downloads/data/cycles_base/ --label picoquic_iblt_30ms_hint --proxy sidekick --freq-pkts 16 --freq-ms 10 --threshold 160 --riblt --quack-hint --network-statistics picoquic --client-quacker --ack-delay 30 -n 25000000
./deps/build_deps.sh 9
sudo -E python3 emulation/main.py --bw1 20 --bw2 20 --delay2 100 --loss1 4 --loss2 0 -t 1 --logdir /home/ubuntu/sidekick-downloads/data/cycles_quack/ --label picoquic_iblt_30ms_hint --proxy sidekick --freq-pkts 16 --freq-ms 10 --threshold 160 --riblt --quack-hint --network-statistics picoquic --client-quacker --ack-delay 30 -n 25000000


In [31]:
# Media benchmark
def gen_media_cmd(label, time_s, base: bool):
    if base:
        print('./deps/build_deps.sh 7')
        home = home_base
    else:
        print('./deps/build_deps.sh 9')
        home = home_quack
    treatment = media_treatment_map[label]
    network_setting = NetworkSetting(bw1=20, bw2=20, delay1=1, delay2=100, loss1=4, loss2=0)
    return RawDataFile(treatment, network_setting, '').cmd(duration=time_s, logdir=home)

print(gen_media_cmd('psum_delay45_hint_nack', 30, True))
print(gen_media_cmd('iblt_delay45_hint_nack', 30, True))
print(gen_media_cmd('psum_delay45_hint_nack', 30, False))
print(gen_media_cmd('iblt_delay45_hint_nack', 30, False))

./deps/build_deps.sh 7
sudo -E python3 emulation/main.py --bw1 20 --bw2 20 --delay2 100 --loss1 4 --loss2 0 --logdir /home/ubuntu/sidekick-downloads/data/cycles_base/ --label psum_delay45_hint_nack --proxy sidekick --freq-ms 0 --threshold 8 --quack-hint --quack-nack --freq-pkts 0 --network-statistics media --client-quacker --ack-delay 45 --duration 30
./deps/build_deps.sh 7
sudo -E python3 emulation/main.py --bw1 20 --bw2 20 --delay2 100 --loss1 4 --loss2 0 --logdir /home/ubuntu/sidekick-downloads/data/cycles_base/ --label iblt_delay45_hint_nack --proxy sidekick --freq-ms 0 --threshold 32 --riblt --quack-hint --quack-nack --freq-pkts 0 --network-statistics media --client-quacker --ack-delay 45 --duration 30
./deps/build_deps.sh 9
sudo -E python3 emulation/main.py --bw1 20 --bw2 20 --delay2 100 --loss1 4 --loss2 0 --logdir /home/ubuntu/sidekick-downloads/data/cycles_quack/ --label psum_delay45_hint_nack --proxy sidekick --freq-ms 0 --threshold 8 --quack-hint --quack-nack --freq-pkts 0 -

## Cycles

In [27]:
def print_router_log(data_home):
    filename = f'{data_home}/router.log'
    with open(filename, 'r') as f:
        print(f.read())

print_router_log(home_base)
print_router_log(home_quack)

Creating logfile /home/ubuntu/sidekick-downloads/data/cycles_base//router.log
Ready to start Sidekick with client p1-eth0; expecting server p1-eth1
INFO [proxy::sidekick::base] Instant { tv_sec: 840109, tv_nsec: 846207739 } Received discovery packet from client. Sidekick: ac10010abe22ac10010b1484, Base: ac10020a1151ac10010ac1aa. Update: false. riblt=true offset=21 threshold=160 cache_policy=Optimistic
INFO [proxy::sidekick::base] Instant { tv_sec: 840109, tv_nsec: 846265134 } Received discovery packet from client. Sidekick: ac10010abe22ac10010b1484, Base: ac10020a1151ac10010ac1aa. Update: true. riblt=true offset=21 threshold=160 cache_policy=Optimistic
INFO [proxy::sidekick::base] Instant { tv_sec: 840109, tv_nsec: 846279190 } Received discovery packet from client. Sidekick: ac10010abe22ac10010b1484, Base: ac10020a1151ac10010ac1aa. Update: true. riblt=true offset=21 threshold=160 cache_policy=Optimistic
connection_type,forward,handle_base_packet_from_server,hash_table,cache_add,check_c